In [1]:
%load_ext autoreload
%autoreload 2

# design

> What this assay can see: the vector, the library, and the limits they set.

In [2]:
#| default_exp design

In [3]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail

In [4]:
#| export
import re
import numpy as np
import pandas as pd
from randseq.coding import encode_library, _pattern_window_codes, decode_motif_codes
from randseq.utils import revcomp, get_patterns, _custom_motif_to_regex

In [5]:
#| hide
import os
from randseq.core import find_restricted_motifs
from randseq.utils import calculate_log2fc
from randseq.example_data import get_example_data_dir, get_example_backbone
data_path = get_example_data_dir()

## Can a given motif be detected at all?

A motif is called by contrasting the plasmids that carry it against the plasmids that do not, so
**both groups have to exist and both have to be large enough**. Whether they do is settled before
any data is collected, by the vector and the library.

An enzyme cuts the molecule, not the insert. So write `c_b` for the copies of the site in the
**backbone** and `c_i` for the copies in the **insert**: what a plasmid presents is `c_b + c_i`,
and the question comes in two stages.

### Stage 1 — is the site already in the vector?

If `c_b > 0`, every member of the library carries at least one copy and there is no motif-free
group at all. What that implies is not one thing:

| the system | what a backbone copy does |
|---|---|
| **one site is enough to restrict** — the common case for Type I and Type II RM | the backbone already saturates it. `c_i` adds nothing the assay can see and the motif is **effectively invisible** |
| **interference scales with motif density** — as it does for BREX | the contrast comes from `c_i` varying *on top of* `c_b`. Detection still works, and can even **depend** on the backbone copies to bring the plasmid near the threshold |
| **something else limits activity** | removing the backbone copies need not restore a signal at all |

So a backbone copy is a reason to **distrust a negative**, not a proof of one. Nothing below ever
calls a motif undetectable; it reports copy numbers and leaves the reading to you.

### Stage 2 — if the vector is clean, is the split big enough?

With `c_b = 0` the contrast is between `c_i = 0` and `c_i >= 1`, and that is set by the insert
alone. The probability that a random insert of length `L` contains a motif with `k` defined
bases, on either strand, is `1 - (1 - 4**-k)**(2*(L-k+1))`:

In [6]:
def carrier_fraction(k, L):
    """Chance that a random insert of length `L` carries a given motif with `k` defined
    bases, on either strand."""
    return 1 - (1 - 4.0 ** -k) ** (2 * (L - k + 1))

def library_shape(counts):
    "Library size and typical insert length, read off a count table."
    return len(counts), int(pd.Series([len(s) for s in counts.index]).median())

ks = range(3, 14)
tables = {}
for f in ["countsTable.csv.gz", "counts_panel.csv.gz"]:
    N, L = library_shape(pd.read_csv(os.path.join(data_path, f), index_col=0))
    p = np.array([carrier_fraction(k, L) for k in ks])
    tables[f"{N:,} inserts of {L} nt"] = pd.DataFrame(
        {"carriers": np.round(p * N).astype(int),
         "non-carriers": np.round((1 - p) * N).astype(int)},
        index=pd.Index(ks, name="defined bases"))

pd.concat(tables, axis=1)      # the two libraries that ship with the package

11,629 inserts of 20 nt  ... 34,217 inserts of 150 nt
                             carriers  ...             non-carriers
defined bases                          ...                         
3                                5032  ...                      323
4                                1449  ...                    10827
5                                 358  ...                    25724
6                                  85  ...                    31878
7                                  20  ...                    33621
8                                   5  ...                    34068
9                                   1  ...                    34180
10                                  0  ...                    34208
11                                  0  ...                    34215
12                                  0  ...                    34216
13                                  0  ...                    34217

[11 rows x 4 columns]

Read each pair of columns together — **both** have to stay large.

- **Too few defined bases and almost every plasmid carries the motif.** There is no motif-free
  baseline left, so even a site that destroys every plasmid carrying it comes back at the
  background rate, indistinguishable from no signal.
- **Too many and almost none does**, which is not enough support to tell a real effect from
  chance.

Where that usable window falls is a property of the library, not of the method, and the two
shipped libraries put it in different places. The 150 nt library runs from about `k = 5` to about
`k = 9`, which covers both architectures `get_patterns()` searches: the contiguous Type II sites
of 5–7 defined bases, and the bipartite Type I sites of 6–8 defined bases split by a spacer. The
20 nt library, a third the size with inserts a seventh as long, is down to 20 carriers at
`k = 7` and 5 at `k = 8`. It can still call a 6 bp Type II site; the 8 defined bases of a
bipartite Type I site are beyond it, so a negative for one of those means nothing at all.
Lengthening the insert raises the top of the window and erodes the bottom; enlarging the library
raises the top alone.

Run `carrier_fraction` on your own `N` and `L` before reading anything into a motif you did not
find.

Note that stage 1 bites hardest exactly where stage 2 is most comfortable. A short site is easy
to detect from the insert and almost certain to be in the vector already; a long bipartite site
is the reverse. The two stages are not independent, and the next section measures the first one.

## What the backbone hides

Stage 1 is a property of the vector, so it can be answered exactly once the vector sequence is
known — no data required.

`pMin` is a 2.2 kb mobilisable vector. The library is cloned in place of a `ccdB` cassette, so
what every library member shares is the 2,214 bp running from one side of the insert round to the
other, which is what ships as `pMin_backbone.fasta`.

In [7]:
backbone = get_example_backbone()
print(f"{len(backbone)} bp; starts at the right flank, ends at the left flank")
print(f"  {backbone[:20]} ... {backbone[-20:]}")

2214 bp; starts at the right flank, ends at the left flank
  ACTAGAGCACCAGAAGTCTA ... GTCTAGGGCGGCGGTAAAAC


In [8]:
#| export
def _motif_site_count(sequence, motif, circular=False):
    """How many times `motif` occurs as a site in `sequence`, on either strand.

    A site is counted once however it reads, which is what an enzyme sees, so a palindromic
    motif is not counted twice. `circular=True` also finds sites spanning the origin.
    """
    seq = sequence.upper()
    if circular and len(seq) >= len(motif):
        seq = seq + seq[:len(motif) - 1]
    pat = _custom_motif_to_regex(motif)
    n = len(re.findall(f"(?={pat})", seq))
    rc_motif = revcomp(motif)
    if rc_motif != motif:                       # not a palindrome: the other reading is distinct
        n += len(re.findall(f"(?={_custom_motif_to_regex(rc_motif)})", seq))
    return n


def backbone_motif_counts(backbone, patterns, circular=False):
    """Every motif of `patterns` that the backbone already carries, and how many copies.

    `circular=False` is right for a library backbone, which is a linear stretch running from one
    side of the insert round to the other -- the insert closes it, so it is not circular on its
    own. Pass `circular=True` for a whole plasmid.

    Returns 'motif', 'pattern' and 'copies', one row per motif carried at least once.
    """
    rows = []
    for pattern in patterns:
        d1, spacer, d2 = pattern
        width = d1 + spacer + d2
        seq = backbone.upper()
        if circular and len(seq) >= width:
            seq = seq + seq[:width - 1]
        if len(seq) < width:
            continue
        f, r = encode_library([seq])
        n_codes = 4 ** (d1 + d2)
        counts = np.zeros(n_codes + 1, dtype=np.int64)
        for arr in (f, r):
            codes = _pattern_window_codes(arr, pattern).ravel()
            counts += np.bincount(codes, minlength=n_codes + 1)
        present = np.flatnonzero(counts[:n_codes])
        if not len(present):
            continue
        for motif, c in zip(decode_motif_codes(present, pattern), counts[present]):
            # a palindrome reads identically on both strands, so each site was counted twice
            rows.append(dict(motif=motif, pattern=str(pattern),
                             copies=int(c) // 2 if revcomp(motif) == motif else int(c)))
    return pd.DataFrame(rows, columns=["motif", "pattern", "copies"])


def backbone_masking_summary(backbone, patterns, circular=False):
    """Per pattern, how much of the searchable motif space the backbone already occupies.

    A motif the backbone carries is one the assay is unlikely to report for this vector, so this
    is the coverage of the search space -- the share of possible answers ruled out before any
    data is collected.
    """
    counts = backbone_motif_counts(backbone, patterns, circular=circular)
    carried = counts.groupby("pattern").size() if len(counts) else pd.Series(dtype=int)
    rows = []
    for pattern in patterns:
        space = 4 ** (pattern[0] + pattern[2])
        n = int(carried.get(str(pattern), 0))
        rows.append(dict(pattern=str(pattern), defined_bases=pattern[0] + pattern[2],
                         motifs=space, carried=n, fraction=n / space))
    return pd.DataFrame(rows)


def annotate_backbone(results_df, backbone, circular=False):
    """Add `backbone_copies` to a result table: copies of each reported motif in the backbone.

    0 is the expected value for anything the pipeline reported, since a motif the backbone
    carries is usually not detectable in the first place. A non-zero count is worth looking at:
    it means the motif was called *despite* the vector already carrying it, which happens when
    interference is density-dependent.
    """
    if results_df is None or len(results_df) == 0 or "motif" not in results_df.columns:
        return results_df
    out = results_df.copy()
    out["backbone_copies"] = [_motif_site_count(backbone, m, circular) for m in out["motif"]]
    return out

In [9]:
#| hide
# counted on both strands, once per site
test_eq(_motif_site_count("AAAGGTCTCAAA", "GGTCTC"), 1)
test_eq(_motif_site_count("AAAGAGACCAAA", "GGTCTC"), 1)        # the same site, other strand
test_eq(_motif_site_count("GGTCTCAAGAGACC", "GGTCTC"), 2)
test_eq(_motif_site_count("AAAAAAAA", "GGTCTC"), 0)
# a palindrome reads identically both ways and is one site, not two
test_eq(_motif_site_count("AAAGGTACCAAA", "GGTACC"), 1)
# IUPAC and N are honoured
test_eq(_motif_site_count("AAACACTTTTGTATAAA", "CACNNNNGTAT"), 1)
test_eq(_motif_site_count("AAACACTTTTGTACAAA", "CACNNNNGTAY"), 1)
# circular only when asked: this site spans the origin
test_eq(_motif_site_count("CTCAAAAAAAAAGGT", "GGTCTC", circular=False), 0)
test_eq(_motif_site_count("CTCAAAAAAAAAGGT", "GGTCTC", circular=True), 1)

In [10]:
#| hide
# the scan-based counter and the regex-based one must agree, motif for motif
_bb = "".join(np.random.default_rng(0).choice(list("ACGT"), 400))
_c = backbone_motif_counts(_bb, [(6, 0, 0), (3, 4, 4)])
assert len(_c) and (_c.copies > 0).all()
for _r in _c.sample(25, random_state=0).itertuples():
    test_eq(_r.copies, _motif_site_count(_bb, _r.motif))
# a backbone with no room for the pattern yields nothing rather than raising
test_eq(len(backbone_motif_counts("ACGT", [(6, 0, 0)])), 0)
# the summary is just that, counted per pattern
_s = backbone_masking_summary(_bb, [(6, 0, 0)])
test_eq(int(_s.motifs.iloc[0]), 4096)
test_eq(int(_s.carried.iloc[0]), len(_c[_c.pattern == "(6, 0, 0)"]))

On the architectures `get_patterns()` searches:

In [11]:
summary = backbone_masking_summary(backbone, get_patterns())
by_size = (summary.groupby("defined_bases")
                  .agg(patterns=("pattern", "size"), mean_masked=("fraction", "mean")))
print(by_size.to_string(float_format=lambda x: f"{x:.1%}"))
print(f"\nover the whole searched space: {summary.carried.sum():,} of {summary.motifs.sum():,} "
      f"motifs = {summary.carried.sum() / summary.motifs.sum():.1%} already carried")

               patterns  mean_masked
defined_bases                       
5                     9        97.6%
6                    17        62.9%
7                     9        22.8%
8                     8         6.4%

over the whole searched space: 119,913 of 750,592 motifs = 16.0% already carried


**Masking is almost entirely a short-motif problem.** With this vector a 5-base site is a lost
cause — 96% of them are already present — a 6-base site is close to a coin toss, and the long
bipartite Type I sites are nearly all reachable. That is the practical reason the method recovers
Type I specificities more readily than short Type II ones, and it is a property of the vector,
not of the strains.

It also says what a second vector buys: a motif is only truly out of reach if **every** backbone
in use carries it, and two unrelated 2.2 kb vectors share far fewer sites than either carries
alone.

### Reading a result against the backbone

`annotate_backbone` adds the copy number to a result table. Anything the pipeline reports should
normally come back with zero copies, since a motif the vector carries is usually not detectable
in the first place. A non-zero count is not a mistake — it is the density-dependent case, and a
signal to look closer.

In [12]:
_called = pd.DataFrame({"motif": ["AAACNNNNGTC", "GGTAAG"],
                        "num_sequences": [894, 0], "fraction_depleted": [0.96, 0.0]})
annotate_backbone(_called, backbone)

,motif,num_sequences,fraction_depleted,backbone_copies
0,AAACNNNNGTC,894,0.96,0
1,GGTAAG,0,0.00,2


`AAACNNNNGTC` is a motif the panel really calls, and the vector is free of it — which is part of
why it could be called. `GGTAAG` is the BREX site of *E. coli* HS, and the vector carries two
copies: the case where a backbone motif is detected anyway, because interference scales with
density.

## Failure modes, on real strains

The shipped panel exists so that each of these can be run, not just read. See
`example_data/counts_panel_key.csv`.

**Thresholds are not universal.** The defaults were tuned on restriction–modification systems.
BREX interference is weaker: *E. coli* HS has a known BREX system recognising `GGTAAG`, and at
the default `-1.0` the pipeline calls **nothing** for it. At `-0.5` it calls `GGTAAG`. A user
applying the defaults to a BREX strain and concluding "no activity" would be wrong.

**Specificity, demonstrated isogenically.** EDL933 calls `CACNNNNNNNCTGG`; its isogenic RM
knockout `EDL933_dRM` calls nothing. A negative result from an unrelated strain proves much less
than a matched pair.

**A motif already in your vector usually cannot be detected.** That is stage 1 above: every
plasmid carries it, so there is no motif-free baseline. The published screen hit exactly this and
had to change backbone to recover a masked Type I motif, so check your backbone before believing
a negative.

**Library size sets what you can detect.** The expected number of plasmids carrying a given
motif is roughly `n_members × 2 × (insert_len − k + 1) / 4**k`. Long bipartite Type I motifs need
a much larger library than short Type II sites. Below about 100 supporting plasmids a real motif
may simply be missed.

In [13]:
#| eval: false
# The BREX case, run rather than asserted. E. coli HS has a known BREX system
# recognising GGTAAG; the default threshold is too strict to see it.
counts_panel = pd.read_csv(os.path.join(data_path, "counts_panel.csv.gz"), index_col=0)
log2fc_hs = calculate_log2fc(counts_panel[["Control_R1", "HS_R1"]],
                             reference_column="Control_R1", count_threshold=10)["HS_R1"]
left_panel, right_panel = "GTCTAGGGCGGCGGTAAAAC", "ACTAGAGCACCAGAAGTCTA"

for thr in (-1.0, -0.5):
    _fixed, flex = find_restricted_motifs(
        log2fc_hs, left_panel, right_panel,
        flexible_motif_patterns=get_patterns(),
        flexible_motif_log2fc_thr=thr, flexible_motif_score_thr=0.5)
    called = list(flex["motif"]) if flex is not None and not flex.empty else []
    print(f"log2FC threshold {thr}: {called or 'nothing called'}")

--- Starting Fixed-Position Motif Analysis (on original short sequences) ---
Scanning motifs of length 1 to 4 at offsets 0 to 6 from each end, for HS_R1.
Minimum sequence support for a motif: > 5 (i.e., 6 or more)


No motifs met the support and score thresholds.
No core motifs found at fixed positions.

--- Starting Position-Independent Motif Analysis (with context) ---


Scanning patterns:   0%|          | 0/43 [00:00<?, ?it/s]

Scanning patterns:   2%|▏         | 1/43 [00:00<00:11,  3.50it/s]

Scanning patterns:   5%|▍         | 2/43 [00:00<00:11,  3.56it/s]

...

Scanning patterns: 100%|██████████| 43/43 [00:12<00:00,  3.35it/s]

Scanning patterns: 100%|██████████| 43/43 [00:12<00:00,  3.58it/s]

20 candidate(s) across all patterns; 2 remain after redundancy filtering.


1 candidate(s) whose score was borrowed:
    AGGTCTC <- CTTACC (without it: n=13, score=0.46)

1 motifs remaining:
GGTAAG (from pattern (6, 0, 0)), FracDep: 0.69, AvgFC: -0.75, N: 2096


log2FC threshold -0.5: ['GGTAAG']


In [14]:
#| hide
import nbdev; nbdev.nbdev_export()